In [ ]:
# ============================================
# 03_model_training.ipynb
# STEP 6-7: TRAIN MODELS
# ============================================

import numpy as np
import pandas as pd
import scipy.sparse
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("="*50)
print("STEP 6-7: MODEL TRAINING")
print("="*50)

STEP 6-7: MODEL TRAINING


In [ ]:
# ============================================
# Load features
# ============================================

print("\n📂 Loading features...")

# First, make sure we're in the right directory
import os
print("Current directory:", os.getcwd())

# Check if files exist
if os.path.exists('data/X_train_tfidf.npz'):
    print("✅ Files found in 'data' folder")
    X_train = scipy.sparse.load_npz('data/X_train_tfidf.npz')
    X_test = scipy.sparse.load_npz('data/X_test_tfidf.npz')
    y_train = np.load('data/y_train.npy')
    y_test = np.load('data/y_test.npy')
else:
    print("⚠️ Files not found. Creating them now...")

    # Load dataset
    import re
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer

    url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
    df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

    def clean_text(text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['cleaned_message'] = df['message'].apply(clean_text)
    df['label_binary'] = df['label'].map({'spam': 1, 'ham': 0})

    X = df['cleaned_message']
    y = df['label_binary']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    # Save for later use
    import joblib
    import os
    os.makedirs('data', exist_ok=True)
    os.makedirs('models', exist_ok=True)

    joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
    scipy.sparse.save_npz('data/X_train_tfidf.npz', X_train_tfidf)
    scipy.sparse.save_npz('data/X_test_tfidf.npz', X_test_tfidf)
    np.save('data/y_train.npy', y_train.values)
    np.save('data/y_test.npy', y_test.values)

    X_train = X_train_tfidf
    X_test = X_test_tfidf

    print("✅ Features created and saved!")

print(f"\n✅ Features loaded!")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


📂 Loading features...
Current directory: /content
⚠️ Files not found. Creating them now...
✅ Features created and saved!

✅ Features loaded!
X_train shape: (4457, 5000)
X_test shape: (1115, 5000)
y_train shape: (4457,)
y_test shape: (1115,)


In [ ]:
# ============================================
# Train models
# ============================================

results = []

# Function to train and evaluate
def train_and_save(model, name):
    print(f"\n🔹 Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"  ✅ Accuracy:  {acc:.4f}")
    print(f"  ✅ Precision: {prec:.4f}")
    print(f"  ✅ Recall:    {rec:.4f}")
    print(f"  ✅ F1-Score:  {f1:.4f}")

    return {
        'model': name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_pred': y_pred,
        'model_obj': model
    }

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
results.append(train_and_save(lr, "Logistic Regression"))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
results.append(train_and_save(rf, "Random Forest"))

# 3. SVM
svm = SVC(kernel='linear', random_state=42)
results.append(train_and_save(svm, "SVM"))


🔹 Training Logistic Regression...
  ✅ Accuracy:  0.9722
  ✅ Precision: 1.0000
  ✅ Recall:    0.7919
  ✅ F1-Score:  0.8839

🔹 Training Random Forest...
  ✅ Accuracy:  0.9767
  ✅ Precision: 1.0000
  ✅ Recall:    0.8255
  ✅ F1-Score:  0.9044

🔹 Training SVM...
  ✅ Accuracy:  0.9848
  ✅ Precision: 0.9853
  ✅ Recall:    0.8993
  ✅ F1-Score:  0.9404


In [ ]:
# ============================================
# Model Comparison
# ============================================

print("\n" + "="*50)
print("📊 MODEL COMPARISON SUMMARY")
print("="*50)

comparison = pd.DataFrame(results)
comparison = comparison[['model', 'accuracy', 'precision', 'recall', 'f1']].round(4)
print(comparison.to_string(index=False))

best = comparison.loc[comparison['accuracy'].idxmax()]
print(f"\n🏆 BEST MODEL: {best['model']} with Accuracy: {best['accuracy']:.4f}")


📊 MODEL COMPARISON SUMMARY
              model  accuracy  precision  recall     f1
Logistic Regression    0.9722     1.0000  0.7919 0.8839
      Random Forest    0.9767     1.0000  0.8255 0.9044
                SVM    0.9848     0.9853  0.8993 0.9404

🏆 BEST MODEL: SVM with Accuracy: 0.9848
